# Notebook 2: LSTM-SNP with Fuzzy Feature Augmentation

**Dataset**: SP500

## Description
This notebook implements **fuzzy feature augmentation** for the LSTM-SNP model. A Takagi-Sugeno 
fuzzy inference system processes the current and previous input values to generate an additional 
feature, which is concatenated with the original input before being fed into the unmodified 
LSTM-SNP cell.

The LSTM-SNP cell itself is **NOT modified** — only the input representation is enriched with 
fuzzy-derived features.

## Theory: Fuzzy Feature Augmentation

### LSTM-SNP Cell (Unchanged)
The LSTM-SNP cell equations remain exactly as in the baseline.

### Fuzzy Inference System
A Takagi-Sugeno fuzzy system augments the input:

**Membership Functions** (Fixed Gaussian):
- $\mu_{low}(x) = \exp\left(-\frac{(x - (-1))^2}{2 \cdot 0.5^2}\right)$
- $\mu_{high}(x) = \exp\left(-\frac{(x - (+1))^2}{2 \cdot 0.5^2}\right)$

**Rules** (4 Takagi-Sugeno rules):
1. IF $x(t)$ is low AND $x(t-1)$ is low → $y_1 = a_1 x(t) + b_1 x(t-1) + c_1$
2. IF $x(t)$ is low AND $x(t-1)$ is high → $y_2 = a_2 x(t) + b_2 x(t-1) + c_2$
3. IF $x(t)$ is high AND $x(t-1)$ is low → $y_3 = a_3 x(t) + b_3 x(t-1) + c_3$
4. IF $x(t)$ is high AND $x(t-1)$ is high → $y_4 = a_4 x(t) + b_4 x(t-1) + c_4$

**Defuzzification**: $y_{fuzzy} = \frac{\sum_i w_i y_i}{\sum_i w_i}$ where $w_i = \prod_j \mu_j(x_j)$

**Augmented Input**: $x'(t) = [x(t), y_{fuzzy}(t)]$

## Model Architecture & Implementation

In [1]:
# ============================================================
# ALL IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
from scipy import stats
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.11.0+cu128
NumPy version: 2.5.1
Using device: cuda


### Fuzzy Inference System (NumPy — for preprocessing)

In [2]:
# ============================================================
# Fuzzy Inference System (NumPy — for preprocessing)
#
# Fixed Gaussian membership functions:
#   μ_low(x)  = exp(-(x - (-1))² / (2·0.5²))
#   μ_high(x) = exp(-(x - (+1))² / (2·0.5²))
#
# 4 Takagi-Sugeno rules with fixed consequent parameters:
#   IF x(t) is low  AND x(t-1) is low  → y₁ = 0.5·x(t) + 0.5·x(t-1)
#   IF x(t) is low  AND x(t-1) is high → y₂ = 0.7·x(t) + 0.3·x(t-1) - 0.1
#   IF x(t) is high AND x(t-1) is low  → y₃ = 0.3·x(t) + 0.7·x(t-1) + 0.1
#   IF x(t) is high AND x(t-1) is high → y₄ = 0.5·x(t) + 0.5·x(t-1)
#
# Output: y = Σ(wᵢ·yᵢ) / Σ(wᵢ)
# ============================================================

def gaussian_mf(x, center, sigma=0.5):
    """Fixed Gaussian membership function."""
    return np.exp(-(x - center)**2 / (2 * sigma**2))

def fuzzy_inference_numpy(x_t, x_tm1):
    """
    Compute fuzzy feature from x(t) and x(t-1).
    Uses fixed membership functions and fixed consequent parameters.
    """
    # Membership degrees
    mu_low_xt = gaussian_mf(x_t, center=-1.0)
    mu_high_xt = gaussian_mf(x_t, center=1.0)
    mu_low_xtm1 = gaussian_mf(x_tm1, center=-1.0)
    mu_high_xtm1 = gaussian_mf(x_tm1, center=1.0)

    # Rule firing strengths (product)
    w1 = mu_low_xt * mu_low_xtm1      # low-low
    w2 = mu_low_xt * mu_high_xtm1     # low-high
    w3 = mu_high_xt * mu_low_xtm1     # high-low
    w4 = mu_high_xt * mu_high_xtm1    # high-high

    # Consequent outputs (fixed linear functions)
    y1 = 0.5 * x_t + 0.5 * x_tm1
    y2 = 0.7 * x_t + 0.3 * x_tm1 - 0.1
    y3 = 0.3 * x_t + 0.7 * x_tm1 + 0.1
    y4 = 0.5 * x_t + 0.5 * x_tm1

    # Weighted average defuzzification
    numerator = w1 * y1 + w2 * y2 + w3 * y3 + w4 * y4
    denominator = w1 + w2 + w3 + w4 + 1e-8

    return numerator / denominator

### LSTM-SNP Cell

In [3]:
# ============================================================
# LSTM-SNP Cell (Original — Unmodified)
# ============================================================

class LSTMSNPCell(nn.Module):
    """
    LSTM-SNP Cell: gates r, c, o (hard sigmoid) and generated spikes a (tanh).
    u(t) = r(t)*u(t-1) - c(t)*a(t)
    h(t) = o(t)*a(t)
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.kernel = nn.Linear(input_size, hidden_size * 4, bias=False)
        self.recurrent_kernel = nn.Linear(hidden_size, hidden_size * 4, bias=False)
        self.bias = nn.Parameter(torch.zeros(hidden_size * 4))

        nn.init.xavier_uniform_(self.kernel.weight)
        nn.init.orthogonal_(self.recurrent_kernel.weight)

    def hard_sigmoid(self, x):
        return torch.clamp(0.2 * x + 0.5, 0.0, 1.0)

    def forward(self, x, u_tm1):
        z = self.kernel(x) + self.recurrent_kernel(u_tm1) + self.bias
        z0, z1, z2, z3 = z.chunk(4, dim=-1)

        r = self.hard_sigmoid(z0)   # reset
        c = self.hard_sigmoid(z1)   # consumption
        o = self.hard_sigmoid(z2)   # output/generation
        a = torch.tanh(z3)          # generated spikes

        u = r * u_tm1 - c * a
        h = o * a
        return h, u

### Build Model

In [4]:
# ============================================================
# Model Construction: LSTM-SNP with Fuzzy Feature Augmentation
# input_dim=2: [x(t), y_fuzzy(t)]
# The LSTM-SNP cell is UNMODIFIED.
# ============================================================

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LSTMSNPCell(input_size, hidden_size)
        self.out = nn.Linear(hidden_size, 1)
        self.u = None

    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.hidden_size, device=device)

    def forward(self, x):
        # x is (batch, 1, input_size)
        if self.u is None or self.u.device != x.device:
            self.reset_states(x.size(0), x.device)
        h, self.u = self.cell(x[:, 0, :], self.u)
        return self.out(h)


def build_model_torch(input_dim, units, batch_size=1):
    # batch_size accepted for call-signature compatibility with the Keras
    # build_model, unused here — batch size is set dynamically via reset_states().
    return RNNModel(input_dim, units)

In [5]:
model = build_model_torch(input_dim=2, units=8, batch_size=1).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

RNNModel(
  (cell): LSTMSNPCell(
    (kernel): Linear(in_features=2, out_features=32, bias=False)
    (recurrent_kernel): Linear(in_features=8, out_features=32, bias=False)
  )
  (out): Linear(in_features=8, out_features=1, bias=True)
)

Total params: 361
Trainable params: 361


## Data Pipeline — Dow Jones Industrial Index

In [6]:
series = pd.read_csv(
    r'C:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\dataset\sp500.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()

In [7]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

diff_values = difference(raw_values, 1)
print(f"Differenced data shape: {diff_values.shape}")

Differenced data shape: (250,)


In [8]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

supervised = timeseries_to_supervised(diff_values, 1)
print(f"Supervised data shape: {supervised.shape}")

Supervised data shape: (250, 2)


In [9]:
# ============================================================
# 4. Train-Test Split (Last 60 points as Test)
# ============================================================
train, test = supervised[:-60], supervised[-60:]

print(f"Train: {train.shape}, Test: {test.shape}")

# ============================================================
# 5. Feature Scaling
# ============================================================
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train)  # fit only on train to avoid leakage
train_scaled = scaler.transform(train)
test_scaled = scaler.transform(test)

Train: (190, 2), Test: (60, 2)


In [10]:
# ============================================================
# 6. Reshape for RNN Input + Fuzzy Feature Augmentation
#    x'(t) = [x(t), y_fuzzy(t)],  y_fuzzy(t) = fuzzy_inference_numpy(x(t), x(t-1))
# ============================================================
X_train_raw, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_test_raw,  y_test  = test_scaled[:, 0:-1],  test_scaled[:, -1]

def build_fuzzy_augmented(X_raw, prev_last=0.0):
    """X_raw: (N,1) scaled lag feature. Returns (N,1,2) augmented tensor."""
    X_aug = np.zeros((X_raw.shape[0], 2))
    for i in range(X_raw.shape[0]):
        x_t = X_raw[i, 0]
        x_tm1 = X_raw[i - 1, 0] if i > 0 else prev_last
        X_aug[i, 0] = x_t
        X_aug[i, 1] = fuzzy_inference_numpy(x_t, x_tm1)
    return X_aug.reshape((X_aug.shape[0], 1, 2))

X_train = build_fuzzy_augmented(X_train_raw, prev_last=0.0)
X_test  = build_fuzzy_augmented(X_test_raw,  prev_last=X_train_raw[-1, 0])

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (190, 1, 2), y_train shape: (190,)
X_test shape:  (60, 1, 2)


## Training Loop

In [11]:
#Training loop (60 runs) -- with checkpointing

CHECKPOINT_DIR = "checkpoints_fuzzy_feature_dowjones"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _ckpt_path(run):
    return os.path.join(CHECKPOINT_DIR, f"run_{run}.pt")

def _summary_path():
    return os.path.join(CHECKPOINT_DIR, "summary.pt")

all_rmse = []
all_mse = []
all_nmse = []
all_predictions = []
all_losses = []
all_models = []  # keep every trained model for later noise-robustness eval

print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")
print(f"Checkpoints will be read/written under: {os.path.abspath(CHECKPOINT_DIR)}\n")

N_RUNS = 60
summary_path = _summary_path()

if os.path.exists(summary_path):
    print("Found completed checkpoint summary -> loading, skipping training.")
    saved = torch.load(summary_path, map_location=device, weights_only=False)

    all_models = []
    for sd in saved['model_state_dicts']:
        m = build_model_torch(input_dim=2, units=8).to(device)
        m.load_state_dict(sd)
        m.eval()
        all_models.append(m)

    all_rmse = saved['rmse']
    all_mse = saved['mse']
    all_nmse = saved['nmse']
    all_predictions = saved['predictions']
    all_losses = saved['losses']
    print(f"Loaded {len(all_models)} models from checkpoint.")

else:
    for run in range(N_RUNS):

        run_ckpt_path = _ckpt_path(run)
        if os.path.exists(run_ckpt_path):
            print(f'\n===== RUN {run + 1}/{N_RUNS} (resumed from checkpoint) =====')
            ckpt = torch.load(run_ckpt_path, map_location=device, weights_only=False)

            model = build_model_torch(input_dim=2, units=8).to(device)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()

            all_losses.append(ckpt['run_losses'])
            all_models.append(model)
            all_rmse.append(ckpt['rmse'])
            all_mse.append(ckpt['mse'])
            all_nmse.append(ckpt['nmse'])
            all_predictions.append(ckpt['predictions'])
            print(f"Loaded run {run + 1} -- RMSE: {ckpt['rmse']:.6f}, MSE: {ckpt['mse']:.6f}, NMSE: {ckpt['nmse']:.10f}")
            continue

        print(f'\n===== RUN {run + 1}/{N_RUNS} =====')

        np.random.seed(run)
        torch.manual_seed(run)

        model = build_model_torch(input_dim=2, units=8).to(device)

        # Initialize consumption gate bias to 1.0 (forget gate equivalent)
        with torch.no_grad():
            hs = model.hidden_size
            model.cell.bias.data[hs:2 * hs] = 1.0

        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()

        run_losses = []
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        n_samples = X_train_t.size(0)

        for epoch in range(100):
            model.train()
            model.reset_states(1, device)
            epoch_loss = 0.0
            for i in range(n_samples):
                x_i = X_train_t[i:i + 1]
                y_i = y_train_t[i:i + 1]

                optimizer.zero_grad()
                pred = model(x_i)
                loss = criterion(pred.squeeze(-1), y_i)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                model.u = model.u.detach()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / n_samples
            run_losses.append(avg_loss)
            print(f"Epoch {epoch + 1}/100 completed. Loss: {avg_loss:.6f}")

        all_losses.append(run_losses)
        model.reset_states(1, device)
        all_models.append(model)
        print(f'Training complete for run {run + 1}')

        model.eval()
        with torch.no_grad():
            for i in range(len(X_train)):
                X_input = torch.tensor(X_train[i:i+1], dtype=torch.float32).to(device)
                model(X_input)

        predictions = []
        model.eval()
        with torch.no_grad():
            for i in range(len(X_test)):
                X_input = torch.tensor(X_test[i:i+1], dtype=torch.float32).to(device)
                yhat = model(X_input).item()

                x_val = X_test[i, 0, 0]  # original scaled feature only (2-col scaler)
                new_row = np.array([x_val, yhat]).reshape(1, 2)
                inverted = scaler.inverse_transform(new_row)[0, -1]
                inverted = inverted + raw_values[len(train) + i]
                predictions.append(inverted)

        actual = raw_values[-len(X_test):]
        rmse = sqrt(mean_squared_error(actual, predictions))
        mse = mean_squared_error(actual, predictions)
        meanV = np.mean(actual)
        denominator = np.linalg.norm(np.array(actual) - meanV, 2)
        nmse = mse / np.power(denominator, 2)

        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)

        print(f'Run {run + 1} -- RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

        torch.save({
            'model_state_dict': model.state_dict(),
            'run_losses': run_losses,
            'rmse': rmse,
            'mse': mse,
            'nmse': nmse,
            'predictions': predictions,
        }, run_ckpt_path)

    torch.save({
        'rmse': all_rmse,
        'mse': all_mse,
        'nmse': all_nmse,
        'predictions': all_predictions,
        'losses': all_losses,
        'model_state_dicts': [m.state_dict() for m in all_models],
    }, summary_path)

print(f'\n--- Training Summary (60 runs) ---')
print(f'RMSE: {np.mean(all_rmse):.6f} ± {np.std(all_rmse):.6f}')
print(f'MSE:  {np.mean(all_mse):.6f} ± {np.std(all_mse):.6f}')
print(f'NMSE: {np.mean(all_nmse):.10f} ± {np.std(all_nmse):.10f}')


--- [PyTorch] RUNNING ON cuda ---

Checkpoints will be read/written under: c:\Users\paulp\OneDrive\Desktop\Type_2\Type_2\checkpoints_fuzzy_feature_dowjones

Found completed checkpoint summary -> loading, skipping training.
Loaded 60 models from checkpoint.

--- Training Summary (60 runs) ---
RMSE: 24.380609 ± 0.172296
MSE:  594.443804 ± 8.398140
NMSE: 0.0027525867 ± 0.0000388878


## Gaussian Noise-Robustness Sweep (0.5% / 5% / 10% / 15%)

Same protocol as the SNN-Transformer / LSTM-SNP noise-robustness notebooks:

`x_noisy(t) = x(t) + eps(t)`,  `eps(t) ~ N(0, sigma_eps^2)`,  `sigma_eps = eta * std(x)`

Noise is injected into the **input features only** (never the targets), using the 60 models
already trained for each sigma above (`sigma_results[sigma]['models']`) -- no retraining is
needed. Each (sigma, noise level) combination is evaluated over multiple noise draws and
reported as a mean +/- std over the 60 runs, exactly mirroring the clean-data summary.

In [12]:
# ============================================================
# Gaussian noise injection
# ============================================================
ref_std = X_train.std()
print(f"Reference std(x) used for noise scaling (train inputs): {ref_std:.6f}")

NOISE_LEVELS = [0.005, 0.05, 0.10, 0.15]  # 0.5%, 5%, 10%, 15%
N_NOISE_DRAWS = 10  # average over multiple eps(t) realizations per model, per level

def add_gaussian_noise(X, noise_level, ref_std, seed=None):
    """
    x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = noise_level * std(x)
    Applied to input features X only -- never to targets/labels.
    """
    rng = np.random.default_rng(seed)
    sigma_eps = noise_level * ref_std
    eps = rng.normal(loc=0.0, scale=sigma_eps, size=X.shape)
    return X + eps

Reference std(x) used for noise scaling (train inputs): 0.211295


In [13]:
# ============================================================
# Evaluation helper -- warms the recurrent state up on clean training data,
# then predicts on X_eval (clean or Gaussian-noise-corrupted, fuzzy-augmented
# test inputs, shape (N,1,2)). Test targets are always the clean,
# ground-truth values.
# ============================================================

def evaluate_on_test(model, X_eval, train, raw_values, scaler, device):
    model.eval()
    model.reset_states(1, device)

    with torch.no_grad():
        for i in range(len(X_train)):
            X_input = torch.tensor(X_train[i:i+1], dtype=torch.float32).to(device)
            model(X_input)

        predictions = []
        for i in range(len(X_eval)):
            X_input = torch.tensor(X_eval[i:i+1], dtype=torch.float32).to(device)
            yhat = model(X_input).item()

            x_val = X_eval[i, 0, 0]  # original (possibly noisy) scaled feature only
            new_row = np.array([x_val, yhat]).reshape(1, 2)
            inverted = scaler.inverse_transform(new_row)[0, -1]
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)

    actual = raw_values[-len(X_eval):]
    mse = mean_squared_error(actual, predictions)
    rmse = sqrt(mse)
    meanV = np.mean(actual)
    denominator = np.linalg.norm(np.array(actual) - meanV, 2)
    nmse = mse / np.power(denominator, 2)
    return predictions, rmse, mse, nmse

## Results

In [14]:
print(f'\n{"="*60}')
print(f'  NOISE SWEEP -- FUZZY FEATURE AUGMENTATION MODEL')
print(f'{"="*60}')

clean_rmse, clean_mse, clean_nmse = [], [], []
for model in all_models:
    _, rmse, mse, nmse = evaluate_on_test(model, X_test, train, raw_values, scaler, device)
    clean_rmse.append(rmse)
    clean_mse.append(mse)
    clean_nmse.append(nmse)

print(f"[{'no noise':>10}]  "
      f"RMSE: {np.mean(clean_rmse):.6f} +/- {np.std(clean_rmse):.6f}  |  "
      f"MSE: {np.mean(clean_mse):.6f} +/- {np.std(clean_mse):.6f}  |  "
      f"NMSE: {np.mean(clean_nmse):.10f} +/- {np.std(clean_nmse):.10f}")

noise_robustness = {}
for noise_level in NOISE_LEVELS:
    level_rmse, level_mse, level_nmse = [], [], []

    for run, model in enumerate(all_models):
        draw_rmse, draw_mse, draw_nmse = [], [], []
        for draw in range(N_NOISE_DRAWS):
            seed = hash((noise_level, run, draw)) % (2 ** 32)
            X_test_noisy = add_gaussian_noise(X_test, noise_level, ref_std, seed=seed)
            _, rmse, mse, nmse = evaluate_on_test(model, X_test_noisy, train, raw_values, scaler, device)
            draw_rmse.append(rmse)
            draw_mse.append(mse)
            draw_nmse.append(nmse)

        level_rmse.append(np.mean(draw_rmse))
        level_mse.append(np.mean(draw_mse))
        level_nmse.append(np.mean(draw_nmse))

    noise_robustness[noise_level] = {'rmse': level_rmse, 'mse': level_mse, 'nmse': level_nmse}
    print(f"[{noise_level * 100:5.1f}% noise]  "
          f"RMSE: {np.mean(level_rmse):.6f} +/- {np.std(level_rmse):.6f}  |  "
          f"MSE: {np.mean(level_mse):.6f} +/- {np.std(level_mse):.6f}  |  "
          f"NMSE: {np.mean(level_nmse):.10f} +/- {np.std(level_nmse):.10f}")


  NOISE SWEEP -- FUZZY FEATURE AUGMENTATION MODEL
[  no noise]  RMSE: 19.823272 +/- 0.942335  |  MSE: 393.850125 +/- 36.401431  |  NMSE: 0.0023356104 +/- 0.0002158678
[  0.5% noise]  RMSE: 19.823319 +/- 0.942363  |  MSE: 393.852016 +/- 36.402562  |  NMSE: 0.0023356216 +/- 0.0002158745
[  5.0% noise]  RMSE: 19.825096 +/- 0.942720  |  MSE: 393.923411 +/- 36.421939  |  NMSE: 0.0023360450 +/- 0.0002159894
[ 10.0% noise]  RMSE: 19.823674 +/- 0.940819  |  MSE: 393.864814 +/- 36.327785  |  NMSE: 0.0023356975 +/- 0.0002154311
[ 15.0% noise]  RMSE: 19.825613 +/- 0.945389  |  MSE: 393.951616 +/- 36.534553  |  NMSE: 0.0023362123 +/- 0.0002166572


In [ ]:
print('===== ROBUSTNESS SUMMARY (60 runs per condition, averaged over 10 noise draws) =====')
print(f"{'Condition':<12}{'RMSE':>14}{'MSE':>16}{'NMSE':>16}")
print(f"{'Clean':<12}{np.mean(clean_rmse):>14.6f}{np.mean(clean_mse):>16.6f}{np.mean(clean_nmse):>16.10f}")
for noise_level in NOISE_LEVELS:
    r = noise_robustness[noise_level]
    label = f"{noise_level * 100:.1f}% noise"
    print(f"{label:<12}{np.mean(r['rmse']):>14.6f}{np.mean(r['mse']):>16.6f}{np.mean(r['nmse']):>16.10f}")

print('\n===== PAIRED DEGRADATION (per-model delta vs its own clean RMSE) =====')
for noise_level in NOISE_LEVELS:
    delta = np.array(noise_robustness[noise_level]['rmse']) - np.array(clean_rmse)
    t_stat, p_val = stats.ttest_rel(noise_robustness[noise_level]['rmse'], clean_rmse)
    print(f"{noise_level*100:5.1f}% noise — ΔRMSE: {delta.mean():.6f} ± {delta.std():.6f}  "
          f"(paired t-test p={p_val:.4f})")

## Observations

### Gaussian Noise robustness on Dow Jones Industrial Index

**Run the notebook to populate results.**